# LAVKA WBD Incremental Loader

Инкрементальная загрузка данных Лавки WBD из SharePoint → `ecom_sandbox`.  
6 таблиц: 5 метрик (`ao`, `orders`, `osa`, `prediction`, `sales_stock`) + `directory`.  
В каждой: `metric_version` (дата из имени файла) + `_source_file` (имя файла).

In [ ]:
import os, re, shutil
from datetime import datetime

import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.utils import AnalysisException

notebook_path = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook().getContext().notebookPath().get()
)
team_folder = notebook_path.split("/")[2]

import sys
sys.path.append(
    f"/Workspace/eperfectstore-prod/{team_folder}/notebooks/"
    f"eperfectstore-prod/e-com/COMMON_FUNCTIONS_AND_CONSTANTS_FOLDER/"
)
import common_functions_and_constants as CF

In [ ]:
SP_BASE = "https://pepsico.sharepoint.com/teams/AzureCCplatform/"


def parse_date_from_filename(filename):
    """Ищет 8-значную дату (ddmmyyyy или yyyymmdd) в имени файла."""
    m = re.search(r'(?<!\d)(\d{8})(?!\d)', os.path.basename(filename))
    if not m:
        return None
    raw = m.group()
    for fmt in ("%d%m%Y", "%Y%m%d"):
        try:
            return datetime.strptime(raw, fmt).date()
        except ValueError:
            pass
    return None


def normalize_col_name(name):
    """'Some Column Name!' → 'some_column_name'"""
    s = re.sub(r"[^0-9a-zA-Z]+", "_", name.strip().lower())
    return re.sub(r"_+", "_", s).strip("_")


def cleanup_and_download(sp_path, temp_path):
    """Очищает temp-папку и скачивает файлы из SharePoint."""
    if os.path.exists(temp_path):
        for name in os.listdir(temp_path):
            p = os.path.join(temp_path, name)
            if os.path.isfile(p) or os.path.islink(p):
                os.remove(p)
            elif os.path.isdir(p):
                shutil.rmtree(p)
    else:
        os.makedirs(temp_path, exist_ok=True)
    CF.copy_from_spo(SP_BASE, sp_path, temp_path)


def get_new_files(temp_path, target_table):
    """
    Список файлов для загрузки.
    Если таблица существует — только файлы с датой > max(metric_version).
    Иначе — все файлы.
    """
    all_files = sorted([
        f for f in os.listdir(temp_path)
        if os.path.isfile(os.path.join(temp_path, f)) and not f.startswith("~$")
    ])

    try:
        max_ver = spark.table(target_table).select(F.max("metric_version")).first()[0]
    except AnalysisException:
        max_ver = None

    if max_ver is None:
        return all_files

    return [
        f for f in all_files
        if (dt := parse_date_from_filename(f)) is not None
        and pd.to_datetime(dt) > pd.to_datetime(max_ver)
    ]


def write_incremental(pdf_list, target_table):
    """Concat pandas DFs → Spark DF → append в target_table."""
    if not pdf_list:
        print(f"{target_table}: нет новых файлов")
        return
    sdf = spark.createDataFrame(pd.concat(pdf_list, ignore_index=True))
    sdf.write.mode("append").option("mergeSchema", "true").saveAsTable(target_table)
    print(f"{target_table}: +{sdf.count()} rows")

## WBD Metrics (5 таблиц)

In [ ]:
WBD_METRICS = [
    ("Shared Documents/WBD/Ecom/Lavka/CPFR_metrics/metrics_ao/",
     "/dbfs/ecom/lavka/wbd/metrics/metrics_ao",
     "ecom_sandbox.lavka_wbd_ao"),

    ("Shared Documents/WBD/Ecom/Lavka/CPFR_metrics/metrics_orders/",
     "/dbfs/ecom/lavka/wbd/metrics/metrics_orders",
     "ecom_sandbox.lavka_wbd_orders"),

    ("Shared Documents/WBD/Ecom/Lavka/CPFR_metrics/metrics_osa/",
     "/dbfs/ecom/lavka/wbd/metrics/metrics_osa",
     "ecom_sandbox.lavka_wbd_osa"),

    ("Shared Documents/WBD/Ecom/Lavka/CPFR_metrics/metrics_prediction/",
     "/dbfs/ecom/lavka/wbd/metrics/metrics_prediction",
     "ecom_sandbox.lavka_wbd_prediction"),

    ("Shared Documents/WBD/Ecom/Lavka/CPFR_metrics/metrics_sales_stock/",
     "/dbfs/ecom/lavka/wbd/metrics/metrics_sales_stock",
     "ecom_sandbox.lavka_wbd_sales_stock"),
]

for sp_path, temp, target in WBD_METRICS:
    cleanup_and_download(sp_path, temp)
    files = get_new_files(temp, target)

    dfs = []
    for fname in files:
        pdf = pd.read_csv(os.path.join(temp, fname))
        pdf.columns = [normalize_col_name(c) for c in pdf.columns]

        if "date" in pdf.columns:
            pdf["date"] = pd.to_datetime(pdf["date"], errors="coerce")

        mv = parse_date_from_filename(fname)
        pdf["metric_version"] = pd.to_datetime(mv) if mv else pd.NaT
        pdf["_source_file"] = fname
        dfs.append(pdf)

    write_incremental(dfs, target)

## WBD Directory

In [ ]:
SP_PATH = "Shared Documents/WBD/Ecom/Lavka/CPFR_metrics/directory/"
TEMP    = "/dbfs/ecom/lavka/wbd/directory"
TARGET  = "ecom_sandbox.lavka_wbd_directory"

cleanup_and_download(SP_PATH, TEMP)

pdf = pd.read_csv(os.path.join(TEMP, "products.csv"))
pdf.columns = [normalize_col_name(c) for c in pdf.columns]
pdf["metric_version"] = pd.NaT
pdf["_source_file"] = "products.csv"

sdf = spark.createDataFrame(pdf)
sdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TARGET)
print(f"{TARGET}: overwritten, {sdf.count()} rows")